# **Adaptive Mulit-Agent RAG**

In [ ]:
# Central Configuration Dictionary to manage all system parameters
config = {
    "data_dir": "./data",                           # Directory to store raw and cleaned data
    "vector_store_dir": "./vector_store",           # Directory to persist our vector store
    "llm_provider": "openai",                       # The LLM provider we are using
    "reasoning_llm": "gpt-4o",                      # The powerful model for planning and synthesis
    "fast_llm": "gpt-4o-mini",                      # A faster, cheaper model for simpler tasks like the baseline RAG
    "embedding_model": "text-embedding-3-small",    # The model for creating document embeddings
    "reranker_model": "cross-encoder/ms-marco-MiniLM-L-6-v2", # The model for precision reranking
    "max_reasoning_iterations": 7,                  # A safeguard to prevent the agent from getting into an infinite loop
    "top_k_retrieval": 10,                          # Number of documents for initial broad recall
    "top_n_rerank": 3,                              # Number of documents to keep after precision reranking
}

In [ ]:
# importing all necessary libraries
from dotenv import load_dotenv
import os
import json
import uuid
import re
from pprint import pprint
from typing import List, Literal, Dict, Optional, TypedDict

# loading all environment variables
load_dotenv()

In [ ]:
import requests
from bs4 import BeautifulSoup
    
# Custom function downloading the 10-K filing and parsing the raw HTML, and converting it into a clean and structured text format suitable for ingestion by RAG pipeline

def download_and_parse_10k(url, data_path_raw, data_path_clean):
    # check if cleaned url already exits to avoid redownloading of dataset
    if os.path.exists(data_path_clean):
        print(f'Cleaned data already exitst at {data_path_clean}')
        return 
    
    # downloading 10k-filling
    print(f"Downloading 10k-filing from url {url}")
    headers = {'User-Agent': 'Mozilla/5.0', }
    respose = requests.get(headers=headers, url=url)
    respose.raise_for_status()

    with open(data_path_raw, 'w', encoding='utf-8') as f:
        f.write(respose.text)
    print(f"Raw data saved to {data_path_raw}")

    soup = BeautifulSoup(respose.content, 'html.parser')

    text = ''
    for i in soup.find_all(['p', 'div', 'span']):
        text += i.get_text(strip=True) + '\n\n'

